## 1. Setup

Install from the repository root with `python -m pip install -r requirement.txt`, then select that Python environment as the notebook kernel. Restart the kernel after changing CAAF source files. This demo imports CAAF from the current repository. Preprocessing always loads the full dataset. An optional commented smoke-test cell is provided in the Ranking section.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Locate repository data from the notebook directory or repository root.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Airfoil_lift_prediction" / "airfoil_5_CL.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Run this notebook from the CAAF repository or its Demo directory.")
sys.path.insert(0, str(root))
import CAAF

from Demo.utils.airfoil_data_processing import load_airfoil_data


## 2. Preprocessing

Use cubic interpolation at timestep 0.0025, pressure times from every fourth lift timestamp starting at index 3, and the `cylinder_5` spatial grid. Concatenate wall 1 then wall 2, scale each case by centered range, and repeat the third case three times (`[0, 0, 2]` augmentation). The headerless lift CSVs retain their first data row, and each wall uses its own source coordinates. These correct the time shift and wall-2 interpolation in the historical reference notebook, so rankings can differ.


In [ ]:
# Preprocessing uses 30,000 samples per case, producing 150,000 rows and 376 sensors.
X, y, wall1, wall2 = load_airfoil_data(root / "Airfoil_lift_prediction",
                                     samples_per_case=30000)
assert X.shape == (150000, 376)
assert np.isfinite(X).all() and np.isfinite(y).all()
print(f"Pressure shape: {X.shape}; lift shape: {y.shape}")

## 3. Ranking

CAAF clusters the full supplied histories, trains on representatives, and averages integrated gradients across runs. Preprocessing already supplies normalized data, so the zero IG baseline represents each case's mean pressure, not zero physical pressure.

This is a sensor-ranking demonstration. The random validation split can contain copies of training samples because the third case is repeated; adjacent time samples are also correlated. Normalization and clustering use all supplied samples. Validation loss is a checkpoint-selection diagnostic, not an independent estimate of predictive accuracy. A predictive benchmark needs separate cases or time blocks held out before normalization, clustering, and augmentation.


The next cell runs the full ranking. For a smoke test, skip that cell and uncomment every line in the optional smoke-test cell below it, then run that cell instead. Continue with the results table and plot. Preprocessing and clustering still use the full dataset; only training and attribution effort are reduced. Set `verbose=False` in either call to silence progress output.

In [ ]:
# Run the complete pipeline once with instantaneous inputs and a zero baseline in the preprocessed coordinates.
sensor_indices, percentages = CAAF.rank_sensors(
    X, y, n_sensors=10, normalization="none",
    clustering={"name": "ap", "preference": None, "damping": 0.5,
                "max_iter": 10000, "convergence_iter": 20},
    model={"name": "mlp", "hidden_sizes": (8, 8, 8), "batch_norm": True,
           "activation": "leaky_relu"},
    sequence={"length": 1, "horizon": 0},
    training={"optimizer": "adam", "lr": 1e-5, "weight_decay": 1e-3,
              "epochs": 100, "batch_size": 64,
              "n_runs": 5, "dtype": "float64",
              "scheduler": {"eps": 1e-7}},
    ig={"n_steps": 15, "batch_size": 512,
        "baseline": "zero", "aggregation": "mean_abs",
        "max_samples": None},
    return_percentages=True,
    verbose=True
)
assert len(sensor_indices) == 10 and len(np.unique(sensor_indices)) == 10
assert np.isfinite(percentages).all() and (percentages >= 0).all()

In [ ]:
# # Run a smoke test on the full preprocessed data with reduced training and attribution.
# sensor_indices, percentages = CAAF.rank_sensors(
#     X, y, n_sensors=10, normalization="none",
#     clustering={"name": "ap", "preference": None, "damping": 0.5,
#                 "max_iter": 10000, "convergence_iter": 20},
#     model={"name": "mlp", "hidden_sizes": (8, 8, 8), "batch_norm": True,
#            "activation": "leaky_relu"},
#     sequence={"length": 1, "horizon": 0},
#     training={"optimizer": "adam", "lr": 1e-5, "weight_decay": 1e-3,
#               "epochs": 2, "batch_size": 64,
#               "n_runs": 1, "dtype": "float64",
#               "scheduler": {"eps": 1e-7}},
#     ig={"n_steps": 3, "batch_size": 512,
#         "baseline": "zero", "aggregation": "mean_abs",
#         "max_samples": 512},
#     return_percentages=True,
#     verbose=True
# )
# assert len(sensor_indices) == 10 and len(np.unique(sensor_indices)) == 10
# assert np.isfinite(percentages).all() and (percentages >= 0).all()

## 4. Results table

Indices are zero-based original pressure columns. Percentages retain their share of attribution across all cluster representatives, so these ten values need not sum to 100%.

In [ ]:
# Align each rank with its original sensor index and attribution percentage.
results = pd.DataFrame({"Rank": np.arange(1, len(sensor_indices) + 1),
                        "Sensor index": sensor_indices, "Attribution (%)": percentages})
results.style.hide(axis="index").format({"Attribution (%)": "{:.3f}"})

## 5. Final sensor configuration

The plot shows the reference airfoil outline and computed sensor locations, annotated by rank.

In [ ]:
# Plot both walls and label computed sensor positions without exporting files.
coordinates = np.vstack((wall1, wall2))
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
for wall in (wall1, wall2):
    ax.plot(wall[:, 0], wall[:, 1], color="0.25", linewidth=1)
selected = coordinates[sensor_indices]
ax.scatter(selected[:, 0], selected[:, 1], color="tab:red", marker="x", s=60, zorder=3)
for rank, (x, y_position) in enumerate(selected, start=1):
    side = 1 if y_position >= 0 else -1
    offset = side * (16 + 13 * ((rank - 1) % 3))
    ax.annotate(str(rank), (x, y_position), xytext=(0, offset),
                textcoords="offset points", ha="center", va="center",
                arrowprops={"arrowstyle": "-", "color": "0.3", "linewidth": 0.6})
ax.set(xlabel="x/C", ylabel="y/C", xlim=(-0.05, 1.05), ylim=(-0.19, 0.19))
ax.set_aspect("equal", adjustable="box")
plt.show()